# Agent 6 — Verification: Try to Kill Every Claim

The verifier reads the sources *hunting for contradictions*; each claim
gets three passes; the majority decides. The vote mechanics run fully
offline — the live cells swap in a real model as the verifier.

**No class API key?** The scripted verifier below IS the lesson; the live
one just replaces its reading with a model's.

In [ ]:
# The mini-web: nine pages about the (fictional) Riverside Community Garden.
# Small enough to read whole, real enough to research. One page is wrong on purpose.
MINIWEB = {
 "riverside-garden.org/about": {"date": "2026-05-10", "title": "Our garden today",
  "text": "The Riverside Community Garden has 60 plots and 48 member families. "
          "We grow vegetables for members and donate surplus to the food pantry."},
 "riverside-garden.org/history": {"date": "2023-05-02", "title": "Our history",
  "text": "Founded in 2019 with a dozen beds. The sign by the gate lists 48 plots, "
          "painted when we finished the 2023 season."},
 "riverside-garden.org/join": {"date": "2026-06-01", "title": "Join us",
  "text": "Want a plot? The waitlist currently holds 22 families. Members pay a "
          "small annual fee and share watering duties."},
 "lakeview-news.com/garden-expands": {"date": "2026-04-20", "title": "Garden adds 12 plots",
  "text": "The Riverside Community Garden completed its expansion this spring, "
          "taking the garden from 48 plots to 60. Organizers credit a city grant."},
 "lakeview-news.com/roundup-2023": {"date": "2023-09-15", "title": "Community roundup",
  "text": "At the Riverside garden, 31 member families closed out the 2023 season "
          "with a harvest festival."},
 "cityparks.gov/report-2026": {"date": "2026-03-14", "title": "Community garden census",
  "text": "Riverside Community Garden: 60 plots, 48 member families, established "
          "2019. Census conducted March 2026."},
 "cityparks.gov/grants-2025": {"date": "2025-11-08", "title": "2025 grant awards",
  "text": "Riverside Community Garden: $15,000 for expansion. The site's land "
          "lease with the parks department runs through 2028."},
 "gardenblog.example.com/visit": {"date": "2026-02-02", "title": "A visit to Riverside",
  "text": "Lovely afternoon at Riverside! I heard they have 600 plots now, which "
          "explains the crowds. The tomatoes were spectacular."},
 "gardenblog.example.com/opinion": {"date": "2026-01-05", "title": "Why gardens matter",
  "text": "Community gardens are the beating heart of a neighborhood. Riverside "
          "is a treasure and everyone loves it."},
}

import re as _re, collections as _c
def _words(text):
    return set(w for w in _re.findall(r"[a-z0-9]+", text.lower()) if len(w) > 2)
_DF = _c.Counter()                       # in how many pages does each word appear?
for _p in MINIWEB.values():
    for _w in _words(_p["title"] + " " + _p["text"]):
        _DF[_w] += 1

def search(query):
    """Score pages by shared words, each weighted by rarity (1/pages-containing-it).
    'riverside' is on every page and says nothing; 'waitlist' is on one and says a lot."""
    qwords = _words(query)
    scored = []
    for url, page in MINIWEB.items():
        shared = qwords & _words(page["title"] + " " + page["text"])
        scored.append((sum(1.0 / _DF[w] for w in shared), url, page["title"]))
    scored.sort(reverse=True)
    return [(url, title) for score, url, title in scored[:3] if score > 0.3]

def fetch(url):
    """Return a page's text with its receipt (url and date) attached."""
    page = MINIWEB[url]
    return {"url": url, "date": page["date"], "text": page["text"]}

print(f"{len(MINIWEB)} pages online.")
print("search('riverside garden plots') ->")
for url, title in search("riverside garden plots"):
    print("  ", url, "-", title)

In [ ]:
%pip install -q anthropic

In [ ]:
import os, getpass
# Ask your teacher for the class API key. It is never typed into a cell,
# never saved in the notebook - getpass keeps it out of your file.
try:
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Class API key: ")
    HAVE_KEY = len(os.environ["ANTHROPIC_API_KEY"]) > 10
except Exception:
    HAVE_KEY = False
print("Key loaded." if HAVE_KEY else "No key - the notebook still teaches: precomputed outputs are shown below each live cell.")

In [ ]:
MODEL = "claude-opus-5"

def ask(prompt, system=None, max_tokens=1000):
    """One model call, plain text in and out."""
    import anthropic
    client = anthropic.Anthropic()
    kwargs = dict(model=MODEL, max_tokens=max_tokens,
                  messages=[{"role": "user", "content": prompt}])
    if system:
        kwargs["system"] = system
    return client.messages.create(**kwargs).content[-1].text

def get_json(prompt, tries=3):
    """Ask for JSON only; parse; re-ask on failure. The retry pattern from Build with LLMs."""
    import json as _json
    for attempt in range(tries):
        text = ask(prompt + "\n\nReply with ONLY valid JSON.")
        try:
            start = text.index("[") if "[" in text.split("{")[0] else text.index("{")
            return _json.loads(text[start:])
        except (ValueError, KeyError):
            continue
    raise RuntimeError("no valid JSON after retries")

## The claim table from lesson 5 (three claims worth watching)

In [ ]:
CLAIMS = [
 {"claim": "Plots grew from 48 (2023) to 60 (2026)",
  "keyfacts": [("48", ["riverside-garden.org/history", "lakeview-news.com/garden-expands"]),
               ("60", ["cityparks.gov/report-2026", "lakeview-news.com/garden-expands",
                       "riverside-garden.org/about"])]},
 {"claim": "The garden has 600 plots",
  "keyfacts": [("600", ["gardenblog.example.com/visit"])]},
 {"claim": "The land lease runs through 2028",
  "keyfacts": [("2028", ["cityparks.gov/grants-2025"])]},
]

## A scripted verifier

One pass = read the sources and try to refute the claim's numbers.
This one is honest arithmetic: for each number, count the pages that
state it and the pages that state a *competing* number for the same
thing. (The live version prompts a model to do this reading; the vote
logic is identical.)

In [ ]:
def one_pass(claim_numbers, competing):
    """Returns (verdict, reason) for one verification pass."""
    votes = []
    for number, supporting_pages in claim_numbers:
        support = len(supporting_pages)
        against = len(competing.get(number, []))
        if against > support:
            votes.append((False, f"{number}: {against} sources contradict, {support} support"))
        elif support == 1 and against == 0:
            votes.append((None, f"{number}: single-source, nothing contradicts"))
        else:
            votes.append((True, f"{number}: {support} sources agree"))
    if any(v is False for v, _ in votes):
        return False, "; ".join(r for v, r in votes if v is False)
    if any(v is None for v, _ in votes):
        return None, "; ".join(r for v, r in votes if v is None)
    return True, "; ".join(r for v, r in votes)

# Competing evidence: pages that state a DIFFERENT current plot count.
COMPETING = {"600": ["cityparks.gov/report-2026", "lakeview-news.com/garden-expands",
                     "riverside-garden.org/about"]}

def verify(claim, passes=3):
    results = [one_pass(claim["keyfacts"], COMPETING) for _ in range(passes)]
    ups = sum(1 for v, _ in results if v is True)
    downs = sum(1 for v, _ in results if v is False)
    if downs >= 2: verdict = "REFUTED"
    elif ups >= 2: verdict = "CONFIRMED"
    else:          verdict = "PLAUSIBLE"
    return verdict, results[0][1]

quarantine = []
for c in CLAIMS:
    verdict, reason = verify(c)
    print(f"{verdict:10} {c['claim']}")
    print(f"           {reason}\n")
    if verdict == "REFUTED":
        quarantine.append({"claim": c["claim"], "reason": reason})

assert verify(CLAIMS[0])[0] == "CONFIRMED"
assert verify(CLAIMS[1])[0] == "REFUTED", "the 600-plot typo must die here"
assert verify(CLAIMS[2])[0] == "PLAUSIBLE"
print("Quarantined (kept, with reasons - never deleted):", quarantine)

## The live verifier — refute, don't confirm

The prompt design that makes model verification real: ask for the
contradiction, not the confirmation.

In [ ]:
VERIFY_PROMPT = """Here is a claim and the full text of every source page.
Your job is to REFUTE the claim if possible: find any source text that
contradicts it. Reply as JSON: {{"verdict": "supported" or "refuted" or
"unsupported", "evidence": "<the contradicting or supporting text>"}}

CLAIM: {claim}

SOURCES:
{sources}"""

if HAVE_KEY:
    sources = "\n".join(f"[{u}] ({p['date']}) {p['text']}" for u, p in MINIWEB.items())
    for c in CLAIMS:
        votes = [get_json(VERIFY_PROMPT.format(claim=c["claim"], sources=sources)) for _ in range(3)]
        print(c["claim"], "->", [v["verdict"] for v in votes])
else:
    print("Precomputed live-run verdicts (3 passes each):")
    print("  Plots grew 48->60      -> ['supported', 'supported', 'supported']")
    print("  The garden has 600 plots -> ['refuted', 'refuted', 'refuted']")
    print("     evidence: 'Riverside Community Garden: 60 plots' (cityparks.gov)")
    print("  Lease runs through 2028 -> ['supported', 'unsupported', 'supported']")

## Try it

1. Change `passes=3` to `passes=1` in the scripted verifier and rerun the
   asserts. (They still pass here because the script is deterministic —
   write one sentence on why a MODEL verifier needs the vote when a
   scripted one doesn't.)
2. **Build turn-in (break the verifier):** plant a second wrong fact in a
   mini-web page, rerun gather → claims → verify, and report whether the
   vote caught your plant — and if not, what property of your edit slipped
   past. Hint: a lie on a page nothing contradicts is the hard case.